# The registered demo, recomputed

Three parameter searches on SPY were fixed in advance (D021) and run once. Their credibility reports are frozen in `results/demo/`. This notebook reads those reports, recomputes every derived figure from the inputs stored next to it with the formulas of [METHODS.md](../docs/METHODS.md), judges each check again with the thresholds of D022, and then tells the story of [SPEC.md](../docs/SPEC.md) Section 3 with numbers taken from the files.

Analyses chosen after seeing the registered results come from `results/posthoc/` and are marked **post hoc** where they appear (D029).

Run it from the repository root with `uv sync --group notebook` and `uv run python notebooks/run.py`; it needs no market data.

In [1]:
import json
import math
from pathlib import Path

import pandas as pd

from backtest_lab.validation import sharpe_stats as ss

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ANNUAL = 252  # trading days per year, as in the reports
SEARCHES = {
    "ma_crossover": "Moving-average crossover",
    "momentum": "Time-series momentum",
    "random_signal": "Random-signal control",
}
reports = {
    key: json.loads((ROOT / "results" / "demo" / f"{key}.json").read_text()) for key in SEARCHES
}
post_hoc = json.loads((ROOT / "results" / "posthoc" / "long_only.json").read_text())
assert post_hoc["post_hoc"] is True

# A recomputed figure must equal the stored one to within this absolute difference. Both
# use the same formulas in double precision; only the order of operations differs.
TOL = 1e-9

## The registered results

SPY from 1993 to October 2026, net of 1 bp fees and 5 bps slippage per side. Sharpe ratios are annualized. The benchmark is buy and hold over the same dates and takes no part in any verdict.

In [2]:
def setting(report):
    params = report["selected"]["params"]
    return ", ".join(f"{name} {params[name]}" for name in report["strategy"]["grid"])


overview = pd.DataFrame(
    {
        "trials": [r["strategy"]["n_trials"] for r in reports.values()],
        "selected": [setting(r) for r in reports.values()],
        "Sharpe": [r["selected"]["metrics"]["sharpe"] for r in reports.values()],
        "buy and hold": [r["benchmark"]["metrics"]["sharpe"] for r in reports.values()],
        "overall": [r["overall"]["verdict"] for r in reports.values()],
    },
    index=list(SEARCHES.values()),
)
overview.round(2)

,trials,selected,Sharpe,buy and hold,overall
Moving-average crossover,128,"fast 10, slow 200, allow_short False",0.80,0.64,yellow
Time-series momentum,34,"lookback 273, allow_short False",0.79,0.65,yellow
Random-signal control,100,seed 55,0.23,0.64,red


In [3]:
checks = pd.DataFrame(
    {
        SEARCHES[key]: {c["title"]: c["verdict"] for c in r["checks"].values()}
        for key, r in reports.items()
    }
)
checks

,Moving-average crossover,Time-series momentum,Random-signal control
Holdout split,green,green,red
Walk-forward analysis,green,green,red
Purged cross-validation,green,green,red
Probability of backtest overfitting,green,yellow,yellow
Deflated Sharpe ratio,yellow,green,red
Probabilistic Sharpe ratio,green,green,yellow
Minimum backtest length,green,green,red
Parameter sensitivity,green,green,red


## Recomputing the figures

Each figure below is computed again from inputs the report stores, and compared with the stored value. The first column names the section of METHODS.md that defines it.

- **PBO (Section 4)** is the share of the 12,870 CSCV splits whose in-sample winner has a non-positive out-of-sample rank logit. The report keeps a 20-bin histogram of the logits; the bins left of zero give the count. A logit of exactly zero counts as overfit but would fall in the bin right of zero, so agreement also shows that no logit landed on zero.
- **DSR (Section 5)** uses the non-annualized Sharpe ratio, skewness, raw kurtosis and length of the selected trial, the variance of the Sharpe ratios across all trials, and the effective number of trials $\hat N = \hat\rho + (1 - \hat\rho) N$ from the average correlation $\hat\rho$.
- **PSR, MinTRL and MinBTL (Section 6)** use the same moments; MinBTL uses $\hat N$ and the selected annualized Sharpe ratio as its target.
- **Retention ratios (Sections 2, 3 and 7)** divide out-of-sample by in-sample performance; the sensitivity ratio is the mean Sharpe ratio of the neighbouring cells of the heat map over the best one.

The PBO itself, the selection in each walk-forward window and fold, and the correlations behind $\hat\rho$ need the matrix of daily trial returns, which is built from market data that is not committed (D003). Those are reproduced by rerunning the search, as the last section describes.

In [4]:
def recompute(report):
    c = report["checks"]
    m = {key: check["metrics"] for key, check in c.items()}
    d = m["deflated_sharpe"]
    n = report["sample"]["n_days"]
    sr = d["sharpe"] / math.sqrt(ANNUAL)  # non-annualized, as the formulas require
    skew, kurt, var = d["skewness"], d["kurtosis"], d["sharpe_variance"]
    n_eff = ss.effective_trials(d["average_correlation"], d["n_trials"])
    logits = c["pbo"]["details"]["logits"]
    below = sum(
        count
        for count, right in zip(logits["counts"], logits["edges"][1:], strict=True)
        if right <= 0
    )
    heat = c["sensitivity"]["details"]["heatmap"]
    near = [heat["sharpe"][cell["y"]][cell["x"]] for cell in heat["neighbours"]]
    best = heat["sharpe"][heat["best"]["y"]][heat["best"]["x"]]
    return [
        (
            "2.1",
            "Holdout retention",
            m["holdout"]["retention"],
            m["holdout"]["oos_sharpe"] / m["holdout"]["is_sharpe"],
        ),
        (
            "2.2",
            "Walk-forward efficiency",
            m["walk_forward"]["efficiency"],
            m["walk_forward"]["oos_return"] / m["walk_forward"]["is_return"],
        ),
        (
            "3",
            "Purged CV retention",
            m["purged_cv"]["retention"],
            m["purged_cv"]["cv_sharpe"] / m["purged_cv"]["train_sharpe"],
        ),
        ("4", "PBO", m["pbo"]["pbo"], below / m["pbo"]["n_combinations"]),
        ("5", "Effective trials", d["n_effective"], n_eff),
        (
            "5",
            "Expected best Sharpe by luck",
            d["expected_max_sharpe"],
            ss.expected_max_sharpe(n_eff, var) * math.sqrt(ANNUAL),
        ),
        ("5", "DSR", d["dsr"], ss.deflated_sharpe_ratio(sr, var, n_eff, n, skew, kurt)),
        (
            "6",
            "PSR(0)",
            m["track_record"]["psr"],
            ss.probabilistic_sharpe_ratio(sr, 0.0, n, skew, kurt),
        ),
        (
            "6",
            "MinTRL, years",
            m["track_record"]["min_track_record_years"],
            ss.min_track_record_length(sr, 0.0, skew, kurt, m["track_record"]["alpha"]) / ANNUAL,
        ),
        (
            "6",
            "MinBTL, years",
            m["min_backtest_length"]["min_backtest_years"],
            ss.min_backtest_length(n_eff, m["min_backtest_length"]["target_sharpe"]),
        ),
        ("7", "Sensitivity retention", m["sensitivity"]["retention"], sum(near) / len(near) / best),
    ]


rows = [
    (SEARCHES[key], section, name, stored, again)
    for key, report in reports.items()
    for section, name, stored, again in recompute(report)
]
figures = pd.DataFrame(rows, columns=["search", "METHODS", "figure", "stored", "recomputed"])
figures["difference"] = (figures["recomputed"] - figures["stored"]).abs()
assert (figures["difference"] <= TOL).all(), figures[figures["difference"] > TOL]
print(f"{len(figures)} figures agree; the largest difference is {figures['difference'].max():.1e}")

33 figures agree; the largest difference is 0.0e+00


In [5]:
figures.set_index(["search", "figure"])[["METHODS", "stored", "recomputed"]].round(6)

METHODS      stored  \
search                   figure                                             
Moving-average crossover Holdout retention                2.1    1.030539   
                         Walk-forward efficiency          2.2    0.611240   
                         Purged CV retention                3    0.864425   
                         PBO                                4    0.047164   
                         Effective trials                   5   47.936359   
                         Expected best Sharpe by luck       5    0.515685   
                         DSR                                5    0.945150   
                         PSR(0)                             6    0.999997   
                         MinTRL, years                      6    4.384295   
                         MinBTL, years                      6    7.971452   
                         Sensitivity retention              7    0.934999   
Time-series momentum     Holdout retention                2.1    0.979748   
                         Walk-forward efficiency          2.2    0.563968   
                         Purged CV retention                3    0.888516   
                         PBO                                4    0.123543   
                         Effective trials                   5   13.999643   
                         Expected best Sharpe by luck       5    0.388107   
                         DSR                                5    0.987621   
                         PSR(0)                             6    0.999995   
                         MinTRL, years                      6    4.487659   
                         MinBTL, years                      6    4.831854   
                         Sensitivity retention              7    0.975115   
Random-signal control    Holdout retention                2.1   -0.435698   
                         Walk-forward efficiency          2.2   -0.114165   
                         Purged CV retention                3   -1.200155   
                         PBO                                4    0.299378   
                         Effective trials                   5  100.000000   
                         Expected best Sharpe by luck       5    0.408041   
                         DSR                                5    0.147134   
                         PSR(0)                             6    0.907537   
                         MinTRL, years                      6   51.895828   
                         MinBTL, years                      6  123.391023   
                         Sensitivity retention              7   -1.513419   

                                                       recomputed  
search                   figure                                    
Moving-average crossover Holdout retention               1.030539  
                         Walk-forward efficiency         0.611240  
                         Purged CV retention             0.864425  
                         PBO                             0.047164  
                         Effective trials               47.936359  
                         Expected best Sharpe by luck    0.515685  
                         DSR                             0.945150  
                         PSR(0)                          0.999997  
                         MinTRL, years                   4.384295  
                         MinBTL, years                   7.971452  
                         Sensitivity retention           0.934999  
Time-series momentum     Holdout retention               0.979748  
                         Walk-forward efficiency         0.563968  
                         Purged CV retention             0.888516  
                         PBO                             0.123543  
                         Effective trials               13.999643  
                         Expected best Sharpe by luck    0.388107  
                         DSR                             0.987621 

## The verdicts, judged again

The thresholds of D022, fixed before any strategy was run on real data:

| Check | Pass | Borderline | Fail |
|---|---|---|---|
| Holdout, walk-forward, purged CV | ratio at least 0.5 and out of sample positive | ratio below 0.5, out of sample positive | out of sample not positive, or no ratio |
| PBO | up to 0.10 | up to 0.30 | above 0.30 |
| DSR, PSR | from 0.95 | from 0.50 | below 0.50 |
| Minimum backtest length | sample at least MinBTL | | shorter, or Sharpe not positive |
| Sensitivity | retention from 0.75 | from 0.50 | below 0.50, or best Sharpe not positive |

The overall verdict is the worst of the eight.

In [6]:
SEVERITY = {"green": 0, "yellow": 1, "red": 2}


def retention_verdict(oos, ratio):
    if not oos > 0 or ratio is None:
        return "red"
    return "green" if ratio >= 0.5 else "yellow"


def band(value, green, yellow):
    if value is None or value < yellow:
        return "red"
    return "green" if value >= green else "yellow"


def judge(report):
    m = {key: check["metrics"] for key, check in report["checks"].items()}
    pbo = m["pbo"]["pbo"]
    btl = m["min_backtest_length"]
    sens = m["sensitivity"]
    verdicts = {
        "holdout": retention_verdict(m["holdout"]["oos_sharpe"], m["holdout"]["retention"]),
        "walk_forward": retention_verdict(
            m["walk_forward"]["oos_return"], m["walk_forward"]["efficiency"]
        ),
        "purged_cv": retention_verdict(m["purged_cv"]["cv_sharpe"], m["purged_cv"]["retention"]),
        "pbo": "green" if pbo <= 0.10 else "yellow" if pbo <= 0.30 else "red",
        "deflated_sharpe": band(m["deflated_sharpe"]["dsr"], 0.95, 0.5),
        "track_record": band(m["track_record"]["psr"], 0.95, 0.5),
        "min_backtest_length": "green"
        if btl["min_backtest_years"] is not None and btl["years"] >= btl["min_backtest_years"]
        else "red",
        "sensitivity": "red" if not sens["best_sharpe"] > 0 else band(sens["retention"], 0.75, 0.5),
    }
    return verdicts, max(verdicts.values(), key=SEVERITY.__getitem__)


for key, report in reports.items():
    verdicts, overall = judge(report)
    assert verdicts == {k: c["verdict"] for k, c in report["checks"].items()}, key
    assert overall == report["overall"]["verdict"], key
    print(f"{SEARCHES[key]}: {overall}, as registered")

Moving-average crossover: yellow, as registered
Time-series momentum: yellow, as registered
Random-signal control: red, as registered


## The story

### 1. A search with no skill still finds a winner

The random-signal control trades SPY long or short at random every five days; 100 seeds make a search with no skill by construction. Its best seed still has a positive Sharpe ratio, and six of the eight checks fail it.

In [7]:
control = reports["random_signal"]
failed = [c["title"] for c in control["checks"].values() if c["verdict"] == "red"]
assert control["selected"]["metrics"]["sharpe"] > 0 and len(failed) == 6
print(f"Best seed: Sharpe {control['selected']['metrics']['sharpe']:.2f}")
print("Failed:", ", ".join(failed))

Best seed: Sharpe 0.23
Failed: Holdout split, Walk-forward analysis, Purged cross-validation, Deflated Sharpe ratio, Minimum backtest length, Parameter sensitivity


The PBO of the control, 0.299, sits just below the red line of 0.30; about one search in ten with no skill lands there (D022). Its sensitivity verdict compares adjacent seeds, which are unrelated, so it carries no information about the control (D019).

### 2. Both real strategies find that long only beats long/short

Each grid has an `allow_short` axis, and shorting SPY works against the equity premium. The in-sample winner is long only in all but one of the 12,870 CSCV splits (**post hoc**), which keeps the whole-grid PBO low. Both are borderline overall: the moving average narrowly misses a significant DSR, and the momentum PBO is above the green band.

In [8]:
ma, mom = reports["ma_crossover"], reports["momentum"]


def checks_of(key):
    return reports[key]["checks"]


sides = pd.DataFrame(
    {
        SEARCHES[key]: {
            "Sharpe range, long only (post hoc)": "{:.2f} to {:.2f}".format(
                *s["annual_sharpe_range"]["long_only"]
            ),
            "Sharpe range, long/short (post hoc)": "{:.2f} to {:.2f}".format(
                *s["annual_sharpe_range"]["long_short"]
            ),
            "CSCV splits with a long-only winner (post hoc)": "{:,} of {:,}".format(
                round(s["cscv"]["long_only_winner_share"] * s["cscv"]["combinations"]),
                s["cscv"]["combinations"],
            ),
            "PBO, whole grid (registered)": f"{checks_of(key)['pbo']['metrics']['pbo']:.2f}",
            "DSR (registered)": f"{checks_of(key)['deflated_sharpe']['metrics']['dsr']:.3f}",
        }
        for key, s in post_hoc["searches"].items()
    }
)
assert round(ma["checks"]["pbo"]["metrics"]["pbo"], 2) == 0.05
assert round(mom["checks"]["pbo"]["metrics"]["pbo"], 2) == 0.12
assert 0.9445 <= ma["checks"]["deflated_sharpe"]["metrics"]["dsr"] < 0.95
assert ma["overall"]["verdict"] == mom["overall"]["verdict"] == "yellow"
sides

,Moving-average crossover,Time-series momentum
"Sharpe range, long only (post hoc)",0.49 to 0.80,0.54 to 0.79
"Sharpe range, long/short (post hoc)",0.02 to 0.51,0.01 to 0.53
CSCV splits with a long-only winner (post hoc),"12,870 of 12,870","12,869 of 12,870"
"PBO, whole grid (registered)",0.05,0.12
DSR (registered),0.945,0.988


### 3. Among the long-only settings, picking the best is close to a coin flip

**Post hoc.** Restricting each grid to its long-only half, a choice made after seeing which half won, the PBO rises to 0.42 for the moving average and 0.72 for momentum. The best long-only setting still has a significant DSR within that half.

In [9]:
half = pd.DataFrame(
    {
        SEARCHES[key]: {
            "PBO, whole grid (registered)": f"{s['pbo']['full_grid']:.2f}",
            "PBO, long-only half": f"{s['pbo']['long_only_half']:.2f}",
            "settings in the half": str(s["long_only_half_dsr"]["n_trials"]),
            "independent settings": f"{s['long_only_half_dsr']['n_effective']:.1f}",
            "DSR within the half": f"{s['long_only_half_dsr']['dsr']:.4f}",
        }
        for key, s in post_hoc["searches"].items()
    }
)
assert round(post_hoc["searches"]["ma_crossover"]["pbo"]["long_only_half"], 2) == 0.42
assert round(post_hoc["searches"]["momentum"]["pbo"]["long_only_half"], 2) == 0.72
half

,Moving-average crossover,Time-series momentum
"PBO, whole grid (registered)",0.05,0.12
"PBO, long-only half",0.42,0.72
settings in the half,64,17
independent settings,10.5,3.8
DSR within the half,0.9999,1.0000


### 4. A higher Sharpe ratio than buy and hold, from smaller drawdowns

The selected moving-average setting has a higher Sharpe ratio than buy and hold (0.80 against 0.64) because its maximum drawdown is far smaller (−21% against −55%); its CAGR is lower (9.7% against 10.8%).

In [10]:
def profile(metrics):
    return {
        "Sharpe": f"{metrics['sharpe']:.2f}",
        "CAGR": f"{metrics['cagr']:.1%}",
        "max drawdown": f"{metrics['max_drawdown']:.1%}",
        "volatility": f"{metrics['annual_volatility']:.1%}",
    }


against = pd.DataFrame(
    {
        "moving average": profile(ma["selected"]["metrics"]),
        "buy and hold": profile(ma["benchmark"]["metrics"]),
    }
)
assert against.loc["Sharpe"].tolist() == ["0.80", "0.64"]
assert against.loc["CAGR"].tolist() == ["9.7%", "10.8%"]
assert round(ma["selected"]["metrics"]["max_drawdown"], 2) == -0.21
assert round(ma["benchmark"]["metrics"]["max_drawdown"], 2) == -0.55
against

,moving average,buy and hold
Sharpe,0.80,0.64
CAGR,9.7%,10.8%
max drawdown,-20.6%,-55.2%
volatility,12.5%,18.7%


**Post hoc**, over the three deepest buy-and-hold drawdowns:

In [11]:
pd.DataFrame(
    [
        {
            "peak to trough": f"{d['peak']} to {d['trough']}",
            "buy and hold": f"{d['benchmark_return']:.1%}",
            "moving average": f"{d['selected_return']:.1%}",
            "momentum": f"{e['selected_return']:.1%}",
        }
        for d, e in zip(
            post_hoc["searches"]["ma_crossover"]["benchmark_drawdowns"],
            post_hoc["searches"]["momentum"]["benchmark_drawdowns"],
            strict=True,
        )
    ]
)

,peak to trough,buy and hold,moving average,momentum
0,2007-10-09 to 2009-03-09,-55.2%,-9.8%,-15.2%
1,2000-03-24 to 2002-10-09,-47.5%,-10.0%,-13.5%
2,2020-02-19 to 2020-03-23,-33.7%,-15.9%,-21.1%


### 5. Two separate questions

Passing the overfitting checks says a result is unlikely to come from the search alone. Whether it beats the market is a separate question, and each report answers the two apart: `overall` holds the verdict, and `benchmark` with `overall.benchmark_summary` holds the comparison.

In [12]:
print(ma["overall"]["summary"])
print(ma["overall"]["benchmark_summary"])

No check fails, but 1 of 8 are borderline (Deflated Sharpe ratio): treat the selected setting's performance with caution.
The checks ask whether the result could come from the search alone, not whether it beats the market: over the same dates the selected setting's Sharpe ratio was 0.80 against 0.64 for SPY buy and hold, and its annual return 9.7% against 10.8%.


## Running the search again

The registered files are not regenerated (D042). To rerun the three searches on freshly fetched data into another directory:

```bash
uv run python -m backtest_lab.data.update
uv run python -m backtest_lab.validation.demo --out /tmp/rerun
```

A rerun on prices fetched on 2026-11-08 gave the same verdicts, selected settings, summaries and PBO values as the registered files. Other numbers moved slightly, all by less than 0.001, because Yahoo had revised its adjusted prices, so the files are not byte-identical ([DEMO.md](../docs/DEMO.md)). The frozen files remain the registered result.